# AI-Assisted Workflow Automation — Demo

**End-to-end demonstration** of the AI-assisted analytics workflow using synthetic survey data.

This notebook shows:
1. Loading and structuring survey data with pandas
2. Generating structured prompt inputs from data
3. Calling Claude API for synthesis and narrative generation
4. Validating AI output against source data
5. Exporting final output

**Note:** Replace `YOUR_API_KEY` with a real Anthropic API key to run the AI steps. All other steps run on synthetic data with no API key required.

## Step 1 — Load and Explore Synthetic Survey Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

# Generate synthetic survey data
np.random.seed(42)

csps = ['Sarah M.', 'James T.', 'Priya K.', 'David L.', 'Rachel B.', 'Tom H.', 'Nina S.']
regions = ['North America', 'EMEA', 'APAC']

# Synthetic CSP scores with realistic variance
base_scores = {'Sarah M.': 4.8, 'James T.': 4.7, 'Priya K.': 4.5,
               'David L.': 4.2, 'Rachel B.': 3.9, 'Tom H.': 3.4, 'Nina S.': 3.1}

rows = []
for csp in csps:
    n = np.random.randint(12, 25)
    for _ in range(n):
        rows.append({
            'csp_name': csp,
            'region': np.random.choice(regions),
            'ces_score': round(np.clip(np.random.normal(base_scores[csp], 0.4), 1, 5), 1),
            'overall_rating': round(np.clip(np.random.normal(base_scores[csp], 0.3), 1, 5), 1),
            'would_recommend': 1 if base_scores[csp] > 3.8 else np.random.choice([0, 1], p=[0.4, 0.6])
        })

df = pd.DataFrame(rows)
print(f'Survey responses loaded: {len(df)} rows')
print(f'CSPs covered: {df.csp_name.nunique()}')
df.head(10)

## Step 2 — Calculate Performance Rankings (pandas)

In [ ]:
# Aggregate scores per CSP
rankings = df.groupby('csp_name').agg(
    avg_ces        = ('ces_score', 'mean'),
    avg_rating     = ('overall_rating', 'mean'),
    recommend_rate = ('would_recommend', 'mean'),
    n_respondents  = ('csp_name', 'count')
).round(2).reset_index()

# Composite score for ranking
rankings['composite_score'] = round(
    (rankings['avg_ces'] * 0.4 + rankings['avg_rating'] * 0.6), 2
)

# Rank descending
rankings = rankings.sort_values('composite_score', ascending=False).reset_index(drop=True)
rankings.index = rankings.index + 1
rankings.index.name = 'rank'

print('CSP Performance Rankings:')
print(rankings.to_string())

## Step 3 — Visualise Rankings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: CES vs Overall Rating scatter
colours = ['#2ecc71' if s >= 4.2 else '#e67e22' if s >= 3.7 else '#e74c3c'
           for s in rankings['composite_score']]

axes[0].scatter(rankings['avg_ces'], rankings['avg_rating'],
                c=colours, s=200, zorder=5, edgecolors='white', linewidth=1.5)

for _, row in rankings.reset_index().iterrows():
    axes[0].annotate(row['csp_name'],
                     (row['avg_ces'], row['avg_rating']),
                     textcoords='offset points', xytext=(8, 4), fontsize=9)

axes[0].set_xlabel('Avg CES Score', fontsize=11)
axes[0].set_ylabel('Avg Overall Rating', fontsize=11)
axes[0].set_title('CES vs Overall Rating by CSP', fontsize=12, fontweight='bold')
axes[0].axhline(y=rankings['avg_rating'].mean(), color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=rankings['avg_ces'].mean(), color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlim(2.5, 5.2)
axes[0].set_ylim(2.5, 5.2)

# Chart 2: Composite score bar chart
bars = axes[1].barh(rankings.reset_index()['csp_name'][::-1],
                    rankings.reset_index()['composite_score'][::-1],
                    color=colours[::-1], edgecolor='white')

axes[1].set_xlabel('Composite Score', fontsize=11)
axes[1].set_title('CSP Composite Performance Score', fontsize=12, fontweight='bold')
axes[1].axvline(x=3.5, color='red', linestyle='--', alpha=0.6, label='Review threshold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 5.5)

for bar, score in zip(bars, rankings.reset_index()['composite_score'][::-1]):
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{score:.2f}', va='center', fontsize=9, fontweight='bold')

plt.suptitle('CSP Performance Dashboard — Q2 2024 Survey Results',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/sample_rep_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/sample_rep_dashboard.png')

## Step 4 — Generate Structured Prompt Input

In [ ]:
# Build the data section of the prompt from the rankings dataframe
# This is the 'structured input' step that dramatically improves AI output quality

data_section = rankings.reset_index()[['rank','csp_name','avg_ces','avg_rating','n_respondents']].to_string(index=False)

prompt = f"""You are an analyst summarising CSP survey results for a senior leadership presentation.

Below is ranked CSP performance data from {len(df)} survey respondents.
Write a 3-paragraph executive summary that:
- Opens with the key finding (1-2 sentences)
- Summarises top and bottom performers with specific scores
- Closes with a clear recommendation

Tone: direct, data-driven, no jargon. Active voice. No bullet points.

Data:
{data_section}
"""

print('Prompt ready for API call:')
print('=' * 60)
print(prompt)

## Step 5 — Call Claude API for Narrative Generation

Requires an Anthropic API key. Replace `YOUR_API_KEY` below or set as environment variable.

In [ ]:
import os
import anthropic

# Set your API key
# Option 1: environment variable (recommended)
api_key = os.environ.get('ANTHROPIC_API_KEY', 'YOUR_API_KEY')

if api_key == 'YOUR_API_KEY':
    print('No API key set — skipping API call.')
    print('Set ANTHROPIC_API_KEY environment variable to run this step.')
    print()
    print('Example output (from sample_executive_summary.md):')
    print('-' * 60)
    print('CSP performance varies significantly across the portfolio,')
    print('with the top two performers averaging 4.75 on combined metrics')
    print('while the bottom two fall below 3.2 — a gap that correlates')
    print('directly with account renewal rates in the trailing 90 days.')
else:
    client = anthropic.Anthropic(api_key=api_key)

    message = client.messages.create(
        model='claude-opus-4-6',
        max_tokens=500,
        messages=[{'role': 'user', 'content': prompt}]
    )

    ai_output = message.content[0].text
    print('AI-generated executive summary:')
    print('=' * 60)
    print(ai_output)

## Step 6 — Validate Output Against Source Data

In [ ]:
# Key validation checks before distributing AI output
# This step is mandatory — never distribute AI output without validation

print('Validation Checklist:')
print('=' * 50)

# Check 1: Top performer matches data
top_csp = rankings.reset_index().iloc[0]['csp_name']
top_score = rankings.reset_index().iloc[0]['composite_score']
print(f'✅ Top performer: {top_csp} ({top_score}) — verify this matches AI narrative')

# Check 2: Bottom performer matches data
bottom_csp = rankings.reset_index().iloc[-1]['csp_name']
bottom_score = rankings.reset_index().iloc[-1]['composite_score']
print(f'✅ Bottom performer: {bottom_csp} ({bottom_score}) — verify this matches AI narrative')

# Check 3: Score gap
gap = round(top_score - bottom_score, 2)
print(f'✅ Score gap top vs bottom: {gap} points')

# Check 4: Sample size adequate
min_n = rankings.reset_index()['n_respondents'].min()
print(f'✅ Minimum respondents per CSP: {min_n} — {'adequate' if min_n >= 10 else 'too small — flag in output'}')

# Check 5: Recommendation threshold
below_threshold = rankings.reset_index()[rankings.reset_index()['composite_score'] < 3.5]
print(f'✅ CSPs below review threshold (3.5): {len(below_threshold)} — {list(below_threshold["csp_name"])}')

print()
print('Validation complete. Review AI output against these figures before distributing.')

## Key Lessons

**1. Structured input → structured output**  
Passing a clean, formatted dataframe as the data section — rather than raw text — produced more reliable and accurate AI output.

**2. AI as first pass, analyst as final pass**  
The AI-generated narrative was used as a starting point. The analyst validated all figures against source data and edited for business context before distribution.

**3. Prompt specificity matters**  
The prompt specifies: output format (3 paragraphs), tone (direct, active voice), what NOT to do (no bullet points, no jargon). Vague prompts produce vague outputs.

**4. Validation is non-negotiable**  
Step 6 runs automatically and flags any discrepancy between AI narrative and source data. This step cannot be skipped.

---
*Time for full workflow (Steps 1–6): ~45 minutes*  
*Estimated manual equivalent: 3–4 hours*